In [1]:
import pandas as pd
import numpy as np

DATA_SOURCE = '../raw-data/XRP.csv'
RETENTION_TIER_SOURCE = '../clean-data/retention_tier.csv'

xrp = pd.read_csv(DATA_SOURCE)
tier = pd.read_csv(RETENTION_TIER_SOURCE)

print("── XRP ──")
print(f"Shape: {xrp.shape}")
print(f"Unique patients: {xrp['PATID'].nunique()}")
print("\nColumns and dtypes:")
print(xrp.dtypes)
print("\nFirst rows:")
print(xrp.head())
print("\nMissing values per column:")
print(xrp.isna().sum())

print("\n── retention_tier ──")
print(f"Shape: {tier.shape}")
print(tier.head())
print(f"\nTier distribution:")
print(tier['retention_tier'].value_counts().sort_index())

── XRP ──
Shape: (6029, 10)
Unique patients: 555

Columns and dtypes:
PROT          int64
PATID         int64
SITE        float64
RANDDT        int64
PROTSEG      object
VISNO        object
XRPASMDT      int64
XRRELAPS      int64
XRRLPCRT    float64
XRPCOMM     float64
dtype: object

First rows:
   PROT   PATID  SITE  RANDDT PROTSEG VISNO  XRPASMDT  XRRELAPS  XRRLPCRT  \
0    51  948442   NaN       0       B    24       189         0       NaN   
1    51  948442   NaN       0       B    20       137         0       NaN   
2    51  948442   NaN       0       B    18       124         0       NaN   
3    51  948442   NaN       0       B    16       109         0       NaN   
4    51  948442   NaN       0       B    12        89         0       NaN   

   XRPCOMM  
0      NaN  
1      NaN  
2      NaN  
3      NaN  
4      NaN  

Missing values per column:
PROT           0
PATID          0
SITE        6029
RANDDT         0
PROTSEG        0
VISNO          0
XRPASMDT       0
XRRELAPS       

In [3]:
# ── Step 2: per-patient aggregation of XRP ─────────────────────────

# Coerce VISNO to integer (currently object due to mixed entries)
xrp['visno_int'] = pd.to_numeric(xrp['VISNO'], errors='coerce')

# Inspect any rows that failed to coerce — likely EOT or screening codes
failed = xrp[xrp['visno_int'].isna()]
print(f"Rows with non-numeric VISNO: {len(failed)}")
if len(failed) > 0:
    print(failed['VISNO'].value_counts())

# Base aggregation: one row per patient
xrp_agg = (
    xrp.groupby('PATID')
    .agg(
        n_xrp_visits   = ('visno_int', 'count'),
        first_xrp_week = ('visno_int', 'min'),
        last_xrp_week  = ('visno_int', 'max'),
        had_relapse    = ('XRRELAPS',  'max'),
        last_asmt_day  = ('XRPASMDT',  'max'),
    )
    .reset_index()
)

# First relapse week (NaN for non-relapsers)
first_relapse = (
    xrp[xrp['XRRELAPS'] == 1]
    .groupby('PATID')['visno_int']
    .min()
    .reset_index()
    .rename(columns={'visno_int': 'first_relapse_week'})
)
xrp_agg = xrp_agg.merge(first_relapse, on='PATID', how='left')

# Relapse criterion (NaN for non-relapsers) — should be constant per patient
relapse_crit = (
    xrp[xrp['XRRELAPS'] == 1]
    .groupby('PATID')['XRRLPCRT']
    .max()
    .reset_index()
    .rename(columns={'XRRLPCRT': 'relapse_criterion'})
)
xrp_agg = xrp_agg.merge(relapse_crit, on='PATID', how='left')

# Mean days between consecutive assessments (engagement texture)
def mean_gap(days):
    s = np.sort(days.values)
    if len(s) < 2:
        return np.nan
    return np.mean(np.diff(s))

gap = (
    xrp.groupby('PATID')['XRPASMDT']
    .apply(mean_gap)
    .reset_index()
    .rename(columns={'XRPASMDT': 'mean_days_between_visits'})
)
xrp_agg = xrp_agg.merge(gap, on='PATID', how='left')

# Inspect
print(f"\nxrp_agg shape: {xrp_agg.shape}")
print(xrp_agg.head())
print("\nSummary:")
print(xrp_agg.describe())
print(f"\nPatients with no relapse: {xrp_agg['first_relapse_week'].isna().sum()}")

Rows with non-numeric VISNO: 2
VISNO
EOT    2
Name: count, dtype: int64

xrp_agg shape: (555, 9)
   PATID  n_xrp_visits  first_xrp_week  last_xrp_week  had_relapse  \
0    323            13             4.0           19.0            1   
1   3424             4             4.0            7.0            1   
2   3735            10             4.0           16.0            1   
3   3860             1             6.0            6.0            1   
4   4138             1             7.0            7.0            1   

   last_asmt_day  first_relapse_week  relapse_criterion  \
0            135                19.0                1.0   
1             51                 7.0                1.0   
2            114                16.0                1.0   
3             44                 6.0                1.0   
4             51                 7.0                1.0   

   mean_days_between_visits  
0                  8.916667  
1                  7.000000  
2                  9.444444  
3      

In [4]:
# ── Step 3: correlation between retention_tier and XRP-derived variables ─────

from scipy import stats

# Merge tier onto the aggregated XRP features
df = xrp_agg.merge(tier[['PATID', 'retention_tier']], on='PATID', how='inner')
print(f"Merged shape: {df.shape}")  # should be (555, 10)

# Define the features to test, and flag which are sanity checks
features = {
    'n_xrp_visits':             'continuous',
    'first_xrp_week':           'continuous',
    'last_xrp_week':            'continuous_sanity',   # defines the tier
    'last_asmt_day':            'continuous_sanity',   # near-duplicate of last_xrp_week
    'had_relapse':              'binary',
    'first_relapse_week':       'continuous_conditional',  # NaN for non-relapsers
    'relapse_criterion':        'binary_conditional',      # NaN for non-relapsers; 1 vs 2
    'mean_days_between_visits': 'continuous',
}

results = []
for var, kind in features.items():
    sub = df[[var, 'retention_tier']].dropna()
    n = len(sub)

    if kind in ('continuous', 'continuous_sanity', 'continuous_conditional'):
        rho, p = stats.spearmanr(sub[var], sub['retention_tier'])
        results.append({'variable': var, 'type': kind, 'n': n,
                        'test': 'Spearman', 'stat': rho, 'p': p})

    elif kind == 'binary':
        # Point-biserial: appropriate when one var is binary, other is ordinal/continuous
        rho, p = stats.pointbiserialr(sub[var], sub['retention_tier'])
        results.append({'variable': var, 'type': kind, 'n': n,
                        'test': 'Point-biserial', 'stat': rho, 'p': p})

    elif kind == 'binary_conditional':
        # relapse_criterion is 1 or 2 — treat as binary among relapsers
        rho, p = stats.pointbiserialr(sub[var] == 2, sub['retention_tier'])
        results.append({'variable': var, 'type': kind, 'n': n,
                        'test': 'Point-biserial (crit=2 vs 1)', 'stat': rho, 'p': p})

res = pd.DataFrame(results).sort_values('stat', key=abs, ascending=False)
print("\n── Correlations with retention_tier (1–4, engaged patients only) ──")
print(res.to_string(index=False, float_format=lambda x: f'{x:.4f}'))

# Cross-check: tier vs had_relapse contingency
print("\n── Contingency: retention_tier × had_relapse ──")
ct = pd.crosstab(df['retention_tier'], df['had_relapse'], margins=True)
print(ct)

chi2, p, dof, _ = stats.chi2_contingency(ct.iloc[:-1, :-1])
print(f"\nChi-squared: {chi2:.2f}, dof={dof}, p={p:.2e}")

Merged shape: (555, 10)

── Correlations with retention_tier (1–4, engaged patients only) ──
                variable                   type   n                         test    stat      p
           last_xrp_week      continuous_sanity 555                     Spearman  0.9854 0.0000
           last_asmt_day      continuous_sanity 555                     Spearman  0.9498 0.0000
            n_xrp_visits             continuous 555                     Spearman  0.9261 0.0000
      first_relapse_week continuous_conditional 326                     Spearman  0.9214 0.0000
             had_relapse                 binary 555               Point-biserial -0.8350 0.0000
          first_xrp_week             continuous 555                     Spearman -0.5696 0.0000
mean_days_between_visits             continuous 461                     Spearman -0.2774 0.0000
       relapse_criterion     binary_conditional 327 Point-biserial (crit=2 vs 1)  0.1075 0.0522

── Contingency: retention_tier × had_relap

In [5]:
# ── Step 4a: distribution of first_xrp_week, to validate engagement_onset bins ──
print("first_xrp_week distribution among 555 engaged patients:")
print(xrp_agg['first_xrp_week'].value_counts().sort_index())
print(f"\nProposed bins:")
print(f"  on_time (week 4):           {(xrp_agg['first_xrp_week'] == 4).sum()}")
print(f"  slightly_delayed (5–6):     {xrp_agg['first_xrp_week'].between(5, 6).sum()}")
print(f"  late (7+):                  {(xrp_agg['first_xrp_week'] >= 7).sum()}")

first_xrp_week distribution among 555 engaged patients:
first_xrp_week
4.0    437
5.0     22
6.0     78
7.0     17
9.0      1
Name: count, dtype: int64

Proposed bins:
  on_time (week 4):           437
  slightly_delayed (5–6):     100
  late (7+):                  18


In [6]:
# ── Step 4b: landmark feature builder ──

def build_landmark_features(xrp_raw, landmark_week, max_visits_at_landmark):
    """
    Build per-patient features observable at a given landmark week.

    Parameters
    ----------
    xrp_raw : DataFrame
        Raw XRP data with PATID, visno_int, XRRELAPS columns.
    landmark_week : int
        Week at which to truncate (e.g., 8 or 12).
    max_visits_at_landmark : int
        Number of scheduled visits between week 4 and landmark_week inclusive
        (5 for w8: weeks 4,5,6,7,8; 9 for w12: weeks 4-12).

    Returns
    -------
    DataFrame with one row per patient appearing in xrp_raw, columns:
        PATID,
        had_relapse_by_wX        : binary (0/1)
        engagement_onset_wX      : ordinal string
        attendance_density_wX    : proportion (0-1)
    """
    suffix = f"w{landmark_week}"

    # Truncate to rows observable at landmark
    truncated = xrp_raw[xrp_raw['visno_int'] <= landmark_week].copy()

    # ── Feature 1: had_relapse_by_wX ──
    relapse = (
        truncated.groupby('PATID')['XRRELAPS'].max()
        .reset_index()
        .rename(columns={'XRRELAPS': f'had_relapse_by_{suffix}'})
    )

    # ── Feature 2: engagement_onset_wX ──
    onset = (
        truncated.groupby('PATID')['visno_int'].min()
        .reset_index()
        .rename(columns={'visno_int': '_first_week'})
    )

    def classify_onset(w):
        if w == 4:        return 'on_time'
        elif w <= 6:      return 'slightly_delayed'
        else:             return 'late'

    onset[f'engagement_onset_{suffix}'] = onset['_first_week'].apply(classify_onset)
    onset = onset.drop(columns='_first_week')

    # ── Feature 3: attendance_density_wX (proportion) ──
    density = (
        truncated.groupby('PATID')['visno_int'].count()
        .reset_index()
        .rename(columns={'visno_int': '_n_visits'})
    )
    density[f'attendance_density_{suffix}'] = density['_n_visits'] / max_visits_at_landmark
    density = density.drop(columns='_n_visits')

    # ── Merge all three features ──
    features = relapse.merge(onset, on='PATID').merge(density, on='PATID')

    # ── Handle patients with no visits by landmark (only relevant if first_xrp_week > landmark) ──
    # Use the full patient list from xrp_raw to detect them
    all_patients = xrp_raw[['PATID']].drop_duplicates()
    features = all_patients.merge(features, on='PATID', how='left')

    # Patients absent from truncated data: not yet engaged by this landmark
    not_yet_mask = features[f'had_relapse_by_{suffix}'].isna()
    features.loc[not_yet_mask, f'had_relapse_by_{suffix}']     = 0
    features.loc[not_yet_mask, f'engagement_onset_{suffix}']   = 'not_yet_engaged'
    features.loc[not_yet_mask, f'attendance_density_{suffix}'] = 0.0

    features[f'had_relapse_by_{suffix}'] = features[f'had_relapse_by_{suffix}'].astype(int)

    return features


# ── Build both landmark feature tables ──
features_w8  = build_landmark_features(xrp,  landmark_week=8,  max_visits_at_landmark=5)
features_w12 = build_landmark_features(xrp, landmark_week=12, max_visits_at_landmark=9)

print("── Week 8 features ──")
print(f"Shape: {features_w8.shape}")
print(features_w8.head())
print("\nSummary:")
print(features_w8.describe(include='all'))

print("\n── Week 12 features ──")
print(f"Shape: {features_w12.shape}")
print(features_w12.head())
print("\nSummary:")
print(features_w12.describe(include='all'))

# ── Sanity check: feature distributions by retention_tier ──
print("\n── Week 8 features by retention_tier ──")
w8_check = features_w8.merge(tier[['PATID', 'retention_tier']], on='PATID', how='inner')
print(w8_check.groupby('retention_tier').agg({
    'had_relapse_by_w8':     'mean',
    'attendance_density_w8': 'mean',
}))
print("\nengagement_onset_w8 × retention_tier:")
print(pd.crosstab(w8_check['engagement_onset_w8'], w8_check['retention_tier']))

── Week 8 features ──
Shape: (555, 4)
    PATID  had_relapse_by_w8 engagement_onset_w8  attendance_density_w8
0  948442                  0             on_time                    1.0
1   85038                  0             on_time                    1.0
2  626492                  0             on_time                    1.0
3  632936                  0             on_time                    1.0
4  415576                  0             on_time                    1.0

Summary:
                PATID  had_relapse_by_w8 engagement_onset_w8  \
count      555.000000         555.000000                 555   
unique            NaN                NaN                   4   
top               NaN                NaN             on_time   
freq              NaN                NaN                 437   
mean    490131.641441           0.329730                 NaN   
std     289857.222466           0.470539                 NaN   
min        323.000000           0.000000                 NaN   
25%     

In [7]:

# ── Sanity check: feature distributions by retention_tier ──
print("\n── Week 12 features by retention_tier ──")
w12_check = features_w12.merge(tier[['PATID', 'retention_tier']], on='PATID', how='inner')
print(w12_check.groupby('retention_tier').agg({
    'had_relapse_by_w12':     'mean',
    'attendance_density_w12': 'mean',
}))
print("\nengagement_onset_w12 × retention_tier:")
print(pd.crosstab(w12_check['engagement_onset_w12'], w12_check['retention_tier']))


── Week 12 features by retention_tier ──
                had_relapse_by_w12  attendance_density_w12
retention_tier                                            
1                         0.981707                0.201220
2                         0.657407                0.602881
3                         0.032787                0.872495
4                         0.000000                0.933433

engagement_onset_w12 × retention_tier:
retention_tier         1    2   3    4
engagement_onset_w12                  
late                  17    1   0    0
on_time               62  101  57  217
slightly_delayed      85    6   4    5


In [10]:
# ── Step 5: ordinal-encode engagement_onset and save feature tables ──

ONSET_ORDINAL = {
    'on_time':          0,
    'slightly_delayed': 1,
    'late':             2,
    'not_yet_engaged':  3,   # patient had no visits by the landmark week
}

OUTPUT_DIR = '../clean-data'

def encode_and_save(features, landmark_week, output_dir):
    suffix = f"w{landmark_week}"
    col    = f"engagement_onset_{suffix}"
    out    = features.copy()

    # Map ordinal codes; unmapped values become NaN, which we detect as a safety check
    out[col] = out[col].map(ONSET_ORDINAL)

    if out[col].isna().any():
        unmapped = features.loc[out[col].isna(), col].unique()
        raise ValueError(f"Unmapped engagement_onset values at {suffix}: {unmapped}")

    out[col] = out[col].astype(int)

    path = f"{output_dir}/features_{suffix}.csv"
    out.to_csv(path, index=False)
    print(f"Saved {path}  shape={out.shape}")
    print(out.head())
    print(f"\n{col} distribution:")
    print(out[col].value_counts().sort_index())
    print()
    return out

features_w8_enc  = encode_and_save(features_w8,  8,  OUTPUT_DIR)
features_w12_enc = encode_and_save(features_w12, 12, OUTPUT_DIR)

Saved ../clean-data/features_w8.csv  shape=(555, 4)
    PATID  had_relapse_by_w8  engagement_onset_w8  attendance_density_w8
0  948442                  0                    0                    1.0
1   85038                  0                    0                    1.0
2  626492                  0                    0                    1.0
3  632936                  0                    0                    1.0
4  415576                  0                    0                    1.0

engagement_onset_w8 distribution:
engagement_onset_w8
0    437
1    100
2     17
3      1
Name: count, dtype: int64

Saved ../clean-data/features_w12.csv  shape=(555, 4)
    PATID  had_relapse_by_w12  engagement_onset_w12  attendance_density_w12
0  948442                   0                     0                0.777778
1   85038                   0                     0                1.000000
2  626492                   0                     0                1.000000
3  632936                   0      